# Phase-2 Step 3: Baseline Models (CORRECTED)

## Important Fix

The original experiment used **cross-battery split** (train on B0005, test on B0006/B0007), which tests generalization but NOT model validity.

This corrected version uses **within-battery time-split**:
- Train on first 70% of ALL batteries
- Test on last 15% of ALL batteries

This properly validates that our models can learn SOH prediction before testing cross-battery generalization.

**Why negative R² before?**
- Different batteries have different degradation patterns
- Model trained on B0005 doesn't generalize to B0006/B0007
- This is expected behavior - cross-battery generalization is hard!

In [ ]:
# --- Portable setup (Windows / macOS / Linux) — do not hardcode paths ---
import importlib.util
from pathlib import Path

def _load_notebook_setup():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "notebooks" / "notebook_setup.py",
        cwd / "notebook_setup.py",
    ]
    if cwd.name == "notebooks":
        candidates.insert(0, cwd / "notebook_setup.py")
    for fp in candidates:
        if fp.is_file():
            spec = importlib.util.spec_from_file_location("_notebook_setup", fp)
            mod = importlib.util.module_from_spec(spec)
            assert spec.loader is not None
            spec.loader.exec_module(mod)
            return mod
    raise FileNotFoundError(
        "Missing notebooks/notebook_setup.py. Unzip the full project, open a terminal, "
        "cd into the repo root, then run: jupyter notebook"
    )

_ns = _load_notebook_setup()
PROJECT_ROOT = _ns.setup()
print(f"PROJECT_ROOT = {PROJECT_ROOT}")


import sys
import os

# ============================================================================
# SETUP
# ============================================================================


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from baseline_models import (
    LSTMModel, CNNLSTMModel, TransformerModel, MLPBaseline,
    SOHFeatureDataset, set_seed
)
from data_splits import load_and_prepare_data
from utils import plot_style_setup

plot_style_setup()

with open('phase2_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

set_seed(config['random_seed'])

print("✓ Imports complete")
print(f"✓ Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 1. Load Data with WITHIN-BATTERY Split

In [ ]:
# Load with proper within-battery time split
data = load_and_prepare_data(PROJECT_ROOT, split_type='within_battery')

train_df = data['train_df']
val_df = data['val_df']
test_df = data['test_df']
feature_cols = data['feature_cols']

print(f"\nFeatures: {len(feature_cols)}")
print(f"\nTrain SOH: [{train_df['soh'].min():.2f}, {train_df['soh'].max():.2f}]")
print(f"Val SOH:   [{val_df['soh'].min():.2f}, {val_df['soh'].max():.2f}]")
print(f"Test SOH:  [{test_df['soh'].min():.2f}, {test_df['soh'].max():.2f}]")

In [ ]:
# Prepare tensors
X_train = train_df[feature_cols].values.astype(np.float32)
y_train = train_df['soh'].values.astype(np.float32)

X_val = val_df[feature_cols].values.astype(np.float32)
y_val = val_df['soh'].values.astype(np.float32)

X_test = test_df[feature_cols].values.astype(np.float32)
y_test = test_df['soh'].values.astype(np.float32)

print(f"Train: {X_train.shape}")
print(f"Val: {X_val.shape}")
print(f"Test: {X_test.shape}")

## 2. Simple MLP Baseline (No Sequences)

In [ ]:
# Create datasets - using simple feature dataset (no sequences)
# This is more appropriate for the small dataset size

train_dataset = SOHFeatureDataset(X_train, y_train)
val_dataset = SOHFeatureDataset(X_val, y_val)
test_dataset = SOHFeatureDataset(X_test, y_test)

batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
def train_model(model, train_loader, val_loader, epochs=100, patience=15, lr=0.001):
    """Train model with early stopping."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    best_val_loss = float('inf')
    patience_counter = 0
    train_losses = []
    val_losses = []
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            pred = model(x)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)
        train_losses.append(train_loss)
        
        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                pred = model(x)
                val_loss += criterion(pred, y).item()
        val_loss /= len(val_loader)
        val_losses.append(val_loss)
        
        # Early stopping
        if val_loss < best_val_loss - 0.001:
            best_val_loss = val_loss
            patience_counter = 0
            best_state = model.state_dict().copy()
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            print(f"  Early stopping at epoch {epoch+1}")
            break
        
        if (epoch + 1) % 20 == 0:
            print(f"  Epoch {epoch+1}: train={train_loss:.4f}, val={val_loss:.4f}")
    
    model.load_state_dict(best_state)
    return model, train_losses, val_losses


def evaluate_model(model, test_loader):
    """Evaluate model on test set."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()
    
    preds = []
    targets = []
    
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            pred = model(x)
            preds.extend(pred.cpu().numpy())
            targets.extend(y.numpy())
    
    preds = np.array(preds)
    targets = np.array(targets)
    
    mae = np.mean(np.abs(preds - targets))
    rmse = np.sqrt(np.mean((preds - targets)**2))
    ss_res = np.sum((targets - preds)**2)
    ss_tot = np.sum((targets - np.mean(targets))**2)
    r2 = 1 - ss_res / ss_tot
    
    return {'mae': mae, 'rmse': rmse, 'r2': r2, 'preds': preds, 'targets': targets}

In [ ]:
# Train MLP baseline
print("="*60)
print("Training MLP Baseline")
print("="*60)

results = {}

for run in range(3):
    set_seed(config['random_seed'] + run)
    print(f"\nRun {run+1}/3")
    
    model = MLPBaseline(input_dim=len(feature_cols), hidden_dims=[128, 64, 32], dropout=0.2)
    model, train_losses, val_losses = train_model(model, train_loader, val_loader)
    
    metrics = evaluate_model(model, test_loader)
    print(f"  → MAE: {metrics['mae']:.4f}, RMSE: {metrics['rmse']:.4f}, R²: {metrics['r2']:.4f}")
    
    if 'mlp' not in results:
        results['mlp'] = []
    results['mlp'].append(metrics)

In [ ]:
# Summary for MLP
mlp_mae = [r['mae'] for r in results['mlp']]
mlp_rmse = [r['rmse'] for r in results['mlp']]
mlp_r2 = [r['r2'] for r in results['mlp']]

print("\nMLP Results:")
print(f"  MAE:  {np.mean(mlp_mae):.4f} ± {np.std(mlp_mae):.4f}")
print(f"  RMSE: {np.mean(mlp_rmse):.4f} ± {np.std(mlp_rmse):.4f}")
print(f"  R²:   {np.mean(mlp_r2):.4f} ± {np.std(mlp_r2):.4f}")

## 3. Gradient Boosting Baseline (Better for Small Data)

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("="*60)
print("Traditional ML Baselines (better for small datasets)")
print("="*60)

# Combine train and val for sklearn models
X_train_full = np.vstack([X_train, X_val])
y_train_full = np.hstack([y_train, y_val])

# Ridge Regression
print("\n1. Ridge Regression")
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_full, y_train_full)
y_pred_ridge = ridge.predict(X_test)

print(f"  MAE:  {mean_absolute_error(y_test, y_pred_ridge):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_ridge)):.4f}")
print(f"  R²:   {r2_score(y_test, y_pred_ridge):.4f}")

# Random Forest
print("\n2. Random Forest")
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train_full, y_train_full)
y_pred_rf = rf.predict(X_test)

print(f"  MAE:  {mean_absolute_error(y_test, y_pred_rf):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_rf)):.4f}")
print(f"  R²:   {r2_score(y_test, y_pred_rf):.4f}")

# Gradient Boosting
print("\n3. Gradient Boosting")
gb = GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
gb.fit(X_train_full, y_train_full)
y_pred_gb = gb.predict(X_test)

print(f"  MAE:  {mean_absolute_error(y_test, y_pred_gb):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_gb)):.4f}")
print(f"  R²:   {r2_score(y_test, y_pred_gb):.4f}")

## 4. Visualize Results

In [ ]:
# Compare all models
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

models_data = {
    'Ridge': y_pred_ridge,
    'Random Forest': y_pred_rf,
    'Gradient Boosting': y_pred_gb
}

for ax, (name, preds) in zip(axes, models_data.items()):
    ax.scatter(y_test, preds, alpha=0.6, s=40)
    
    min_val = min(y_test.min(), preds.min())
    max_val = max(y_test.max(), preds.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
    
    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    
    ax.set_xlabel('Actual SOH (%)', fontsize=11)
    ax.set_ylabel('Predicted SOH (%)', fontsize=11)
    ax.set_title(f'{name}\nR² = {r2:.4f}, RMSE = {rmse:.2f}', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('Within-Battery Test Results (Proper Split)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/phase2/figures/baseline_corrected_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved: results/phase2/figures/baseline_corrected_comparison.png")

In [ ]:
# Feature importance from Gradient Boosting
importance = gb.feature_importances_
indices = np.argsort(importance)[::-1][:15]  # Top 15

plt.figure(figsize=(10, 6))
plt.barh(range(len(indices)), importance[indices], align='center')
plt.yticks(range(len(indices)), [feature_cols[i] for i in indices])
plt.xlabel('Feature Importance')
plt.title('Top 15 Features for SOH Prediction (Gradient Boosting)', fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('results/phase2/figures/feature_importance_gb.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 10 Features:")
for i, idx in enumerate(indices[:10]):
    print(f"  {i+1}. {feature_cols[idx]}: {importance[idx]:.4f}")

## 5. Summary Table

In [ ]:
# Create summary table
summary = pd.DataFrame([
    {
        'Model': 'Ridge Regression',
        'MAE': mean_absolute_error(y_test, y_pred_ridge),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_ridge)),
        'R²': r2_score(y_test, y_pred_ridge)
    },
    {
        'Model': 'Random Forest',
        'MAE': mean_absolute_error(y_test, y_pred_rf),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_rf)),
        'R²': r2_score(y_test, y_pred_rf)
    },
    {
        'Model': 'Gradient Boosting',
        'MAE': mean_absolute_error(y_test, y_pred_gb),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_gb)),
        'R²': r2_score(y_test, y_pred_gb)
    },
    {
        'Model': 'MLP Neural Net',
        'MAE': np.mean(mlp_mae),
        'RMSE': np.mean(mlp_rmse),
        'R²': np.mean(mlp_r2)
    }
])

print("\n" + "="*70)
print("BASELINE RESULTS: WITHIN-BATTERY TIME SPLIT (CORRECTED)")
print("="*70)
print(summary.to_string(index=False))
print("="*70)

# Best model
best_idx = summary['R²'].idxmax()
best_model = summary.loc[best_idx, 'Model']
best_r2 = summary.loc[best_idx, 'R²']

print(f"\n✓ Best model: {best_model} (R² = {best_r2:.4f})")
print("\nNOTE: These results use within-battery time-split.")
print("Cross-battery generalization (Step 6) will show lower performance.")

# Save
summary.to_csv('results/phase2/baseline_corrected_results.csv', index=False)
print("\n✓ Saved: results/phase2/baseline_corrected_results.csv")

## Summary

**Key Insight:**

The original negative R² values were due to testing **cross-battery generalization** (train on B0005, test on B0006/B0007). This is actually a valid and important test for Step 6.

With **within-battery time-split** (proper validation), we expect:
- R² > 0.8 for good models
- MAE < 3% SOH
- RMSE < 4% SOH

**Evaluation Strategy for Phase 2:**

| Split Type | Purpose | Expected R² |
|------------|---------|-------------|
| Within-Battery | Model validation | 0.8 - 0.95 |
| Cross-Battery | Generalization test | 0.3 - 0.7 (challenging) |

**Next Step:** Proceed to Step 4 with proper splits, then analyze generalization in Step 6.